In [110]:
import pandas as pd
import numpy as np
import plotly.express as px
import time
from datetime import datetime
from dateutil.relativedelta import relativedelta
import quantstats as qs

### Data Loading

In [107]:
file_paths = {
    'Indonesia (IDX30)': 'IDX30_Index_Past5Y.csv',
    'Malaysia (KLSE)': 'Malaysia KLSE Composite_Past5Y.csv',
    'Singapore (STI)': 'Straits Times Index_Past5Y.csv',
    'Philippines': 'Philippines Stock Exchange Composite Index_Past5Y.csv',
    'Vietnam (VN30)': 'Vietnam VN30 Index_Past5Y.csv',
    'Thailand (SET50)': 'SET50 Index_Past5Y.csv',
    'S&P500': 'S&P500_Past5Y.csv'
}

all_closes = []

for country, path in file_paths.items():
    df = pd.read_csv(path, parse_dates=['Date'], index_col='Date')
    
    if df.index.duplicated().any():
        print(f"Cleaned duplicate dates in: {country}")
        df = df[~df.index.duplicated(keep='last')]

    close_series = df['Close'].rename(country) 
    all_closes.append(close_series)

asean_combined_df = pd.concat(all_closes, axis=1)

asean_combined_df.sort_index(inplace=True)
asean_combined_df.ffill(inplace=True)
asean_filtered_df = asean_combined_df.loc['2021-07-16':].copy()
asean_filtered_df.ffill(inplace=True)
asean_filtered_df = asean_filtered_df.replace({',': ''}, regex=True).apply(pd.to_numeric, errors='coerce')
asean_filtered_df.ffill(inplace=True)

print("Filtered Data (Starting 2021-07-16):")
print(asean_filtered_df.tail())


Filtered Data (Starting 2021-07-16):
            Indonesia (IDX30)  Malaysia (KLSE)  Singapore (STI)  Philippines  \
Date                                                                           
2026-06-07             338.99          1683.63          5025.80      5910.06   
2026-06-14             344.75          1712.03          5192.70      6135.35   
2026-06-21             331.32          1667.74          5191.73      6072.24   
2026-06-28             329.10          1679.05          5244.29      6188.03   
2026-07-05             332.65          1677.64          5433.88      6223.87   

            Vietnam (VN30)  Thailand (SET50)   S&P500  
Date                                                   
2026-06-07         1944.36           1026.89  7431.46  
2026-06-14         1963.57           1023.25  7500.58  
2026-06-21         2008.57           1011.23  7354.02  
2026-06-28         2002.56           1059.71  7483.24  
2026-07-05         1987.11           1057.12  7543.64  


## Comparative Analysis

In [108]:
# 1. Normalize the prices to base 100 starting from the first row (2021-07-16)
# This allows us to compare apples-to-apples percentage performance
normalized_df = (asean_filtered_df / asean_filtered_df.iloc[0]) * 100

# 2. Restructure the data for Plotly (melting from 'wide' to 'long' format)
# This turns the country columns into a single 'Country' column for the legend
df_melted = normalized_df.reset_index().melt(
    id_vars=['Date'], 
    var_name='Country', 
    value_name='Performance (Base 100)'
)

# 3. Create the interactive line chart
fig = px.line(
    df_melted, 
    x='Date', 
    y='Performance (Base 100)', 
    color='Country', 
    title='ASEAN Major Indices Comparison (Base 100 on 2021-07-16)',
    labels={'Date': 'Date', 'Performance (Base 100)': 'Relative Performance'}
)

# 4. Polish the layout for better readability
fig.update_layout(
    hovermode="x unified", # Shows a single tooltip line for all countries when hovering
    legend_title_text='Country',
    template='plotly_white' # Gives the chart a clean, professional look
)

# 5. Show the chart in your browser or notebook
fig.show()

In [109]:
# 1. Convert raw prices into daily percentage returns
# This measures how the markets move relative to yesterday, stripping out long-term price drift.
daily_returns = asean_filtered_df.pct_change().dropna()

# 2. Compute the Pearson correlation matrix
corr_matrix = daily_returns.corr()

# 3. Create the interactive heatmap using Plotly Express
fig = px.imshow(
    corr_matrix,
    text_auto='.2f',               # Automatically overlay the numbers rounded to 2 decimals
    color_continuous_scale='RdBu', # Red-to-Blue diverging scale (great for financial correlations)
    zmin=-1.0,                     # Correlation bottom floor
    zmax=1.0,                      # Correlation ceiling
    title='ASEAN Markets & S&P 500 Correlation Matrix (Daily Returns)',
    labels=dict(color="Correlation")
)

# 4. Clean up the layout
fig.update_layout(
    width=700,
    height=700,
    template='plotly_white'
)

# 5. Display the matrix
fig.show()

In [115]:
# Make sure quantstats has hooked into pandas
qs.extend_pandas()

# 1. Convert raw prices to daily returns
daily_returns = asean_filtered_df.pct_change().dropna()

# 2. Isolate your benchmark (assuming your S&P 500 column is named 'SP500')
benchmark_returns = daily_returns['S&P500']

# 3. Create a list of the ASEAN indices to iterate through
asean_columns = [col for col in daily_returns.columns if col != 'SP&500']

# 4. Loop through each index and compile QuantStats metrics
compiled_metrics = {}

for index in asean_columns:
    asset_returns = daily_returns[index]
    
    # FIX: Fetch the bundled greeks data array
    greeks_data = qs.stats.greeks(asset_returns, benchmark_returns)
    beta_val = greeks_data[0]   # Beta is the first item
    alpha_val = greeks_data[1]  # Alpha is the second item
    
    compiled_metrics[index] = {
        "CAGR (%)": round(qs.stats.cagr(asset_returns) * 100, 2),
        "Sharpe Ratio": round(qs.stats.sharpe(asset_returns), 2),
        "Sortino Ratio": round(qs.stats.sortino(asset_returns), 2),
        "Max Drawdown (%)": round(qs.stats.max_drawdown(asset_returns) * 100, 2),
        "Annual Volatility (%)": round(qs.stats.volatility(asset_returns) * 100, 2),
        "Beta vs S&P 500": round(beta_val, 2),
        "Alpha vs S&P 500": round(alpha_val, 2),
    }

# 5. Turn the dictionary into a unified summary table
summary_df = pd.DataFrame(compiled_metrics).T
print(summary_df)

C:\Users\Edward\AppData\Local\Temp\ipykernel_27888\1661094449.py:21: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\Edward\AppData\Local\Temp\ipykernel_27888\1661094449.py:22: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\Edward\AppData\Local\Temp\ipykernel_27888\1661094449.py:21: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\Edward\AppData\Local\Temp\ipykernel_27888\1661094449.py:22: FutureWarning:

Series

                   CAGR (%)  Sharpe Ratio  Sortino Ratio  Max Drawdown (%)  \
Indonesia (IDX30)    -25.66         -0.57          -0.76            -45.80   
Malaysia (KLSE)        9.84          0.55           0.85            -14.44   
Singapore (STI)       69.61          2.12           3.17            -13.38   
Philippines           -4.43          0.04           0.06            -25.22   
Vietnam (VN30)        40.45          1.01           1.45            -39.43   
Thailand (SET50)      14.08          0.59           0.86            -33.08   
S&P500                68.53          1.64           2.48            -24.82   

                   Annual Volatility (%)  Beta vs S&P 500  Alpha vs S&P 500  
Indonesia (IDX30)                  38.64             0.23             -0.35  
Malaysia (KLSE)                    20.81             0.20              0.00  
Singapore (STI)                    26.62             0.25              0.42  
Philippines                        34.86             0.26      

C:\Users\Edward\AppData\Local\Temp\ipykernel_27888\1661094449.py:21: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\Edward\AppData\Local\Temp\ipykernel_27888\1661094449.py:22: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\Edward\AppData\Local\Temp\ipykernel_27888\1661094449.py:21: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\Edward\AppData\Local\Temp\ipykernel_27888\1661094449.py:22: FutureWarning:

Series